In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [2]:
import numpy as np
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

2025-11-27 21:23:47.815437: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764278628.051409      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764278628.115001      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"

In [4]:
img_size = (300, 300) 
batch_size = 32

In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [6]:
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

In [7]:
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [8]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 307 images belonging to 4 classes.


In [9]:
val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 75 images belonging to 4 classes.


In [10]:
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 120 images belonging to 4 classes.


In [11]:
for subdir, _, files in os.walk(train_dir):
    for file in files:
        try:
            img_path = os.path.join(subdir, file)
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            print(f"Corrupted: {img_path} — {e}")

In [12]:
labels = train_gen.classes
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weight = dict(enumerate(class_weight))

In [13]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, verbose=1)

In [14]:
base_model = EfficientNetB3(include_top=False, weights='imagenet', input_shape=(300, 300, 3))
base_model.trainable = False

I0000 00:00:1764278671.553292      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [15]:
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dropout(0.4))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(train_gen.num_classes, activation='softmax'))

In [16]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [17]:
print("phase1..training(freez lower layers)")
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)


phase1..training(freez lower layers)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1764278710.889844     118 service.cc:148] XLA service 0x7d2698002e30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764278710.891066     118 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764278714.104035     118 cuda_dnn.cc:529] Loaded cuDNN version 90300


 2/10 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.1562 - loss: 1.4935 

I0000 00:00:1764278731.241536     118 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 84s 4s/step - accuracy: 0.3334 - loss: 1.3839 - val_accuracy: 0.7733 - val_loss: 0.8385 - learning_rate: 0.0010
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 830ms/step - accuracy: 0.7043 - loss: 0.8389 - val_accuracy: 0.8667 - val_loss: 0.5661 - learning_rate: 0.0010
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 807ms/step - accuracy: 0.8215 - loss: 0.5562 - val_accuracy: 0.8800 - val_loss: 0.4010 - learning_rate: 0.0010
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 810ms/step - accuracy: 0.8547 - loss: 0.4579 - val_accuracy: 0.9067 - val_loss: 0.3499 - learning_rate: 0.0010
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 813ms/step - accuracy: 0.8456 - loss: 0.4561 - val_accuracy: 0.9333 - val_loss: 0.2641 - learning_rate: 0.0010
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 786ms/step - accuracy: 0.9087 - loss: 0.3009 - val_accuracy: 0.8933 - val_loss: 0.2631 - learning_rate: 0.0010
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 780ms/step - accuracy: 0.8922 - loss: 0.3562 - val_accuracy:

In [18]:
print("phase2...finetuning")
base_model.trainable = True
fine_tune_at = 300  
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

phase2...finetuning


In [19]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [20]:
training = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 831ms/step - accuracy: 0.9536 - loss: 0.1636 - val_accuracy: 0.9600 - val_loss: 0.1629 - learning_rate: 1.2500e-04
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 751ms/step - accuracy: 0.9235 - loss: 0.2105 - val_accuracy: 0.9467 - val_loss: 0.1710 - learning_rate: 1.2500e-04
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 724ms/step - accuracy: 0.9472 - loss: 0.1822
Epoch 3: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 766ms/step - accuracy: 0.9466 - loss: 0.1829 - val_accuracy: 0.9467 - val_loss: 0.1784 - learning_rate: 1.2500e-04
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 833ms/step - accuracy: 0.9404 - loss: 0.1799 - val_accuracy: 0.9467 - val_loss: 0.1738 - learning_rate: 6.2500e-05
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 774ms/step - accuracy: 0.9282 - loss: 0.1888
Epoch 5: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 820ms/step - accuracy: 0.9291 - lo

In [21]:
best_train_acc = max(training.history['accuracy'])
best_val_acc = max(training.history['val_accuracy'])

In [22]:
print(f"Best Training Accuracy: {best_train_acc:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

Best Training Accuracy: 0.9414
Best Validation Accuracy: 0.9600


In [23]:
print("Best Validation Accuracy:", max(training.history['val_accuracy']))

Best Validation Accuracy: 0.9599999785423279
